In [ ]:
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd

# Configuración de visualización en Pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

# Definición dinámica de rutas del proyecto
BASE = Path("..") if Path("../data").exists() else Path(".")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# Carga de los datasets originales sin modificarlos
df_lesiones_raw = pd.read_csv(RAW / "lesiones_accidentes_transito.csv")
df_sectores_raw = pd.read_csv(RAW / "sectores_criticos_siniestralidad_vial.csv")

print(f"Dataset Lesiones cargado: {df_lesiones_raw.shape[0]:,} filas x {df_lesiones_raw.shape[1]} columnas")
print(f"Dataset Sectores Críticos cargado: {df_sectores_raw.shape[0]:,} filas x {df_sectores_raw.shape[1]} columnas")

In [ ]:
def normalizar_texto(serie):
    """Elimina espacios extra, convierte a mayúsculas y remueve tildes."""
    return (
        serie.astype(str)
        .str.strip()
        .str.upper()
        .apply(lambda x: "".join(
            c for c in unicodedata.normalize("NFKD", x)
            if not unicodedata.combining(c)
        ))
    )

# --- PIPELINE DE LIMPIEZA: LESIONES ---
def limpiar_lesiones(df):
    df = df.copy()
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df["fecha_hecho"] = pd.to_datetime(df["fecha_hecho"], format="%d/%m/%Y", errors="coerce")
    df["departamento"] = normalizar_texto(df["departamento"])
    df["municipio"] = normalizar_texto(df["municipio"])
    
    dias_es = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}
    df["dia_semana"] = df["fecha_hecho"].dt.dayofweek.map(dias_es)
    df["anio"] = df["fecha_hecho"].dt.year
    return df

def validar_lesiones(df):
    assert df["fecha_hecho"].isna().sum() == 0, "Existen fechas nulas tras conversión"
    assert (df["cantidad"] > 0).all(), "Existen cantidades inválidas"
    return df

df_lesiones_clean = (
    df_lesiones_raw
    .pipe(limpiar_lesiones)
    .pipe(validar_lesiones)
)

# --- PIPELINE DE LIMPIEZA: SECTORES CRÍTICOS ---
def limpiar_sectores(df):
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    for col in ["gizscore", "gipvalue", "latitud", "longitud"]:
        df[col] = df[col].astype(str).str.replace(",", ".", regex=False).astype(float)
    df["tramo"] = df["tramo"].fillna("TRAMO NO ESPECIFICADO").str.strip()
    df["nombre"] = df["nombre"].fillna("SIN NOMBRE REGISTRADO").str.strip()
    df["departamento"] = normalizar_texto(df["departamento"])
    df["municipio"] = normalizar_texto(df["municipio"])
    return df

def validar_sectores(df):
    assert df.isna().sum().sum() == 0, "Persisten nulos en sectores críticos"
    assert df["latitud"].dtype == "float64", "Latitud no es float64"
    return df

df_sectores_clean = (
    df_sectores_raw
    .pipe(limpiar_sectores)
    .pipe(validar_sectores)
)

print("Limpieza y validaciones assert superadas exitosamente.")

In [ ]:
# Filtrado para Medellín (DIVIPOLA 5001) y Área Metropolitana del Valle de Aburrá
df_medellin = df_lesiones_clean[df_lesiones_clean["cod_muni"] == 5001].copy()

municipios_aburra = [
    "MEDELLIN", "BELLO", "ITAGUI", "ENVIGADO", "SABANETA",
    "LA ESTRELLA", "CALDAS", "COPACABANA", "GIRARDOTA", "BARBOSA"
]
df_sectores_aburra = df_sectores_clean[df_sectores_clean["municipio"].isin(municipios_aburra)].copy()

# Indicador 1: Lesionados por día de la semana en Medellín
orden_dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
ind_dia_semana = (
    df_medellin.groupby("dia_semana", as_index=False)
    .agg(eventos=("cantidad", "count"), total_lesionados=("cantidad", "sum"), promedio=("cantidad", "mean"))
)
ind_dia_semana["dia_semana"] = pd.Categorical(ind_dia_semana["dia_semana"], categories=orden_dias, ordered=True)
ind_dia_semana = ind_dia_semana.sort_values("dia_semana")

print("=== INDICADOR 1: LESIONADOS POR DÍA DE LA SEMANA EN MEDELLÍN ===")
display(ind_dia_semana)

# Indicador 2: Evolución anual de lesionados en Medellín (2021 - 2026)
ind_anual = (
    df_medellin[df_medellin["anio"].between(2021, 2026)]
    .groupby("anio", as_index=False)
    .agg(eventos=("cantidad", "count"), total_lesionados=("cantidad", "sum"))
    .sort_values("anio", ascending=False)
)
print("\n=== INDICADOR 2: TENDENCIA ANUAL EN MEDELLÍN (2021-2026) ===")
display(ind_anual)

# Indicador 3: Tramos críticos y fallecidos en el Valle de Aburrá
ind_sectores = (
    df_sectores_aburra.groupby("municipio", as_index=False)
    .agg(tramos_criticos=("id_mt", "count"), total_fallecidos=("fallecidos", "sum"), z_score_medio=("gizscore", "mean"))
    .sort_values("total_fallecidos", ascending=False)
)
print("\n=== INDICADOR 3: SECTORES CRÍTICOS EN EL VALLE DE ABURRÁ ===")
display(ind_sectores)

In [ ]:
# Resumen departamental de Antioquia (cod_depto == 5)
resumen_antioquia = (
    df_lesiones_clean[df_lesiones_clean["cod_depto"] == 5]
    .groupby(["cod_muni", "municipio"], as_index=False)
    .agg(eventos_lesiones=("cantidad", "count"), total_lesionados=("cantidad", "sum"))
)

# Cruce relacional por municipio
df_consolidado = pd.merge(resumen_antioquia, ind_sectores, on="municipio", how="left")
df_consolidado["tramos_criticos"] = df_consolidado["tramos_criticos"].fillna(0).astype(int)
df_consolidado["total_fallecidos"] = df_consolidado["total_fallecidos"].fillna(0).astype(int)

assert len(df_consolidado) == len(resumen_antioquia), "Error: Alteración de registros tras merge"

print("=== TABLA CONSOLIDADA MUNICIPAL (TOP 10 ANTIOQUIA) ===")
display(df_consolidado.sort_values("total_lesionados", ascending=False).head(10))

# Exportación final únicamente en formato CSV hacia data/processed/
df_medellin.to_csv(PROCESSED / "medellin_lesiones_clean.csv", index=False)
df_sectores_clean.to_csv(PROCESSED / "sectores_clean.csv", index=False)
df_consolidado.to_csv(PROCESSED / "municipality_analysis.csv", index=False)

print("\nArchivos CSV exportados exitosamente en 'data/processed/'.")